# Explore the replay store

**What this does:** loads the recent cycle / decision / trade history from the operator's Postgres, summarises by pair / regime / outcome, and plots a few standard views.

**What it doesn't do:** open trades, modify any row, or call an external API. Read-only against `DATABASE_URL`.

Halal alignment: research surface; never feeds back into the cycle. The notebook holds itself to a `commit=False` discipline — the asyncpg connection's transaction never commits.

## 0 · Imports + connection

Pulls the same `Settings.database_url` the bot uses. If you ran `just pg-up` and the `.env` is configured, this cell should connect on the first try.

In [ ]:
from __future__ import annotations

import asyncio
from datetime import datetime, timedelta, UTC

import asyncpg
import pandas as pd

from halal_trader.config import get_settings

settings = get_settings()
DATABASE_URL = settings.database_url
print(f"Connected via {DATABASE_URL.split('@')[-1]}")

## 1 · Recent decisions

The bot writes one `LlmDecision` row per cycle's strategy call. The column we care about most for replay is `parsed_action`, a JSONB blob that contains the per-pair decisions. We aggregate the most-recent N days into a flat DataFrame.

**Tunable:** `LOOKBACK_DAYS` controls how far back to fetch.

In [ ]:
LOOKBACK_DAYS = 14

async def fetch_decisions() -> pd.DataFrame:
    conn = await asyncpg.connect(DATABASE_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT timestamp, provider, model, prompt_version,
                   input_tokens, output_tokens, cost_usd,
                   parsed_action, symbols, execution_ms
            FROM llm_decisions
            WHERE timestamp >= $1
            ORDER BY timestamp DESC
            """,
            datetime.now(UTC) - timedelta(days=LOOKBACK_DAYS),
        )
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

decisions = await fetch_decisions()
print(f"{len(decisions)} decisions in last {LOOKBACK_DAYS} days")
decisions.head()

## 2 · Cost summary

Total LLM spend over the lookback window, broken down by provider + model. Useful for spotting an unexpected cost spike before the daily cap fires.

In [ ]:
if decisions.empty:
    print("No decisions in window.")
else:
    cost = (
        decisions.groupby(["provider", "model"])
        .agg(
            calls=("timestamp", "count"),
            total_usd=("cost_usd", "sum"),
            avg_ms=("execution_ms", "mean"),
        )
        .sort_values("total_usd", ascending=False)
    )
    display(cost.style.format({"total_usd": "${:,.4f}", "avg_ms": "{:,.0f}ms"}))

## 3 · Closed trades

Every `crypto_trade` carries `entry_price` / `exit_price` / `return_pct` for closed rows. We pull only closed trades and compute the per-pair stats.

In [ ]:
async def fetch_closed_trades() -> pd.DataFrame:
    conn = await asyncpg.connect(DATABASE_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT pair, side, quantity,
                   entry_price, exit_price, return_pct,
                   exit_reason, timestamp, closed_at
            FROM crypto_trades
            WHERE closed_at IS NOT NULL
              AND closed_at >= $1
            ORDER BY closed_at DESC
            """,
            datetime.now(UTC) - timedelta(days=LOOKBACK_DAYS),
        )
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

trades = await fetch_closed_trades()
print(f"{len(trades)} closed trades in last {LOOKBACK_DAYS} days")
trades.head()

## 4 · Per-pair performance

Win rate, mean return, total return per pair. The same numbers the dashboard tile shows but with the operator's own filter knobs (date range, pair list, exit reason).

In [ ]:
if trades.empty:
    print("No closed trades in window.")
else:
    trades = trades.copy()
    trades["is_win"] = trades["return_pct"] > 0
    per_pair = (
        trades.groupby("pair")
        .agg(
            trades=("return_pct", "count"),
            win_rate=("is_win", "mean"),
            mean_return=("return_pct", "mean"),
            total_return=("return_pct", "sum"),
        )
        .sort_values("total_return", ascending=False)
    )
    display(
        per_pair.style.format(
            {
                "win_rate": "{:.0%}",
                "mean_return": "{:+.2%}",
                "total_return": "{:+.2%}",
            }
        )
    )

## 5 · Equity curve

Cumulative product of `(1 + return_pct)` over time. The standard backtest output applied to live trades.

In [ ]:
if trades.empty:
    print("No closed trades in window.")
else:
    trades_sorted = trades.sort_values("closed_at")
    trades_sorted["equity"] = (1.0 + trades_sorted["return_pct"].fillna(0)).cumprod()
    ax = trades_sorted.plot(x="closed_at", y="equity", figsize=(12, 4))
    ax.set_title(f"Equity curve over last {LOOKBACK_DAYS} days")
    ax.set_ylabel("equity multiple")
    ax.grid(True, alpha=0.3)

## 6 · Read-only invariant check

This cell is the regression-style proof that the notebook has not modified any row. We compare the row counts of the three tables we read against their counts before the notebook started.

If you see a non-zero delta, that's a bug — open an issue and don't run the notebook against production data until it's fixed.

In [ ]:
async def row_counts() -> dict[str, int]:
    conn = await asyncpg.connect(DATABASE_URL)
    try:
        out = {}
        for table in ("llm_decisions", "crypto_trades", "trades"):
            out[table] = await conn.fetchval(f"SELECT count(*) FROM {table}")
        return out
    finally:
        await conn.close()

post = await row_counts()
print("Post-notebook row counts:", post)
print("(Should match what you saw before running the notebook.)")